# Figure 6 Preparation

In [ ]:
# to create pkl files with angles for certain type 
# run i.e.
# python tools/calculate_angles_cifar.py mean_bias --data_names_file="fig_6.txt"
# available types are:
#     'mean_bias',
#     'only_mean',
#     'only_bias',
#     'nothing'

In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import pickle
import numpy as np

In [ ]:
s = f"""
| Cifar100 | After Conv | After Shifting | $\\Delta$ After Conv | $\\Delta$ After Shifting |
| -- | -- | -- | -- | -- |
"""

layer_names = [
    'After_conv',
    'After_shifting',
    'Before_conv_vs_After_conv',
    'Before_shifting_vs_After_shifting',
]

def format_metrics(avg_metrics):
    labels = ['in', 'out', 'back']
    parts = []

    for i, label in enumerate(labels):
        mean = avg_metrics[2 * i]
        std = avg_metrics[2 * i + 1]
        parts.append(f'{label} {mean:.2f}±{std:.2f}<br>')

    return ''.join(parts)


for type_ in [
    'mean_bias',
    'only_mean',
    'only_bias',
    'nothing's
]:
    PATH = f'heap/cifar100/resnet50_{type_}/'
    cells = []
    for layer_data_name in layer_names:
        with open(PATH + layer_data_name + '.pkl', 'rb') as fr:
            res = pickle.load(fr)
            
        sh = list(res.values())[0]['var'].shape[0]
        mss = np.zeros((sh, sh), dtype=bool)
        np.fill_diagonal(mss[:sh-1, :sh-1], 1)
        mso = np.triu(np.ones((sh, sh), dtype=bool), k=1)
        mso[:, sh-1] = 0
        msb = np.zeros((sh, sh), dtype=bool)
        msb[:sh-1, sh-1] = 1
        
        layer_stats = {}

        for layer_name in res.keys():

            layer_data = res[layer_name]
            metrics = []

            for mask in (mss, mso, msb):
                m_val = np.nanmean(layer_data['mean'][mask])
                s_val = np.sqrt(np.nanmean(layer_data['var'][mask]))
                metrics.extend([m_val, s_val])

            layer_stats[layer_name] = metrics

        all_metrics = np.array(list(layer_stats.values()))
        avg_metrics = np.mean(all_metrics, axis=0)
        cells.append(format_metrics(avg_metrics))

    s += f'''| {type_} | {' | '.join(cells)} |\n'''

print(s)

## Figure 8 Preparation

In [3]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")

import pickle
import numpy as np

from collections import defaultdict

In [ ]:
# to create pkl files with angles for certain type 
# run i.e.
# python tools/calculate_angles_cifar.py mean_bias --data_names_file="fig_8.txt"
# available types are:
#     'mean_bias',
#     'only_mean',
#     'only_bias',
#     'nothing'

In [ ]:
PATH = f'heap/cifar100/resnet50_mean_bias/'

def compute_avg(data):
    res = []
    vals = [np.nanmean(layer['mean'][0])
            for layer in data.values() 
            ]
    res.append(np.mean(vals))
    vals = [np.sqrt(layer['var'][0]) 
            for layer in data.values() 
           ]
    res.append(np.mean(vals))
    return res


VS = [0.05, 0.1, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95]
RES = defaultdict(list)
for v in VS:
    for t in ['top', 'low']:
        for s in ['center', 's']:
            CORR_TYPE = f'{t}_by_{s}_{v}_norm'
            LAYER_DATA_NAME = f'Fig8_While_Conv_{CORR_TYPE}'
            try:
                with open(PATH + LAYER_DATA_NAME + '.pkl', 'rb') as fr:
                        res = pickle.load(fr)
            except:
#                 continue
                raise

            xs, m, s = [], [], []
            a = compute_avg(res)
            m.append(a[0])
            s.append(a[1])
            RES[LAYER_DATA_NAME.replace(str(v), '_')].append((np.nanmean(m), np.nanmean(s)))
            
r = {}
for k, v in RES.items():
    k_ = k.replace('Fig8_While_Conv', '').replace('___norm', '').replace('_', ' ')
    k_ = k_[0].upper() + k_[1:]
    
    r[k_] = ([round(v_[0], 5) for v_ in v], [round(v_[1], 5) for v_ in v])

print(r)